In [ ]:
pip install nest_asyncio langchain nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 660.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Uninstalling tenacity-9.0.0:
      Successfully uninstalled tenacity-9.0.0


In [ ]:
pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.5 MB/s eta 0:00:00


In [ ]:
import requests
from bs4 import BeautifulSoup
import textwrap
import time

# Function to fetch HTML content from a URL with retry mechanism
def fetch_html(url, retries=3):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
    attempt = 0
    while attempt < retries:
        try:
            response = requests.get(url, headers=headers, timeout=20)  # Increased timeout and added User-Agent
            response.raise_for_status()  # Check for request errors
            return response.content
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            attempt += 1
            if attempt < retries:
                print(f"Retrying {url} ({attempt}/{retries})...")
                time.sleep(2)  # Wait before retrying
            else:
                print(f"Failed to fetch {url} after {retries} attempts.")
                return None

# Function to remove header, footer, and extract the remaining content
def extract_main_content(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    # Remove header and footer
    if soup.header:
        soup.header.decompose()
    if soup.footer:
        soup.footer.decompose()

    # Remove page-head element by class
    page_head = soup.find(class_="page-head")
    if page_head:
        page_head.decompose()

    # Remove specific elements by their classes
    specific_classes = [
        "bUqfOz", "hEiKeJ", "gySqrp", "customHeader", "headernavbar", "breadcrumb",
        "customfooter", "content_top", "itr-season-banner", "header-wrapper",
        "common-wrapper", "common-right", "common-left", "container-fluid row",
        "bread_crumbs", "topic", "top-header", "navbar", "nav clearfix",
        "row breadcrumb-outer", "steps px-0", "menu_wrapper", "region region-user-menu",
        "myheadbtnhdr", "top-bar"
    ]

    for class_name in specific_classes:
        elements = soup.find_all(class_=class_name)
        for element in elements:
            element.decompose()

    # Get remaining text from the body
    body_content = soup.body.get_text(separator='\n').strip() if soup.body else ""
    return body_content

# Function to chunk text with overlap
def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(' '.join(chunk))
    return chunks

# List of URLs with meaningful names
urls_with_names = {
     "https://en.wikipedia.org/wiki/Allahabad_High_Court": "Allahabad High Court",
    "https://en.wikipedia.org/wiki/Andhra_Pradesh_High_Court": "Andhra Pradesh High Court",
    "https://en.wikipedia.org/wiki/Bombay_High_Court": "Bombay High Court",
    "https://en.wikipedia.org/wiki/Calcutta_High_Court": "Calcutta High Court",
    "https://en.wikipedia.org/wiki/Chhattisgarh_High_Court": "Chhattisgarh High Court",
    "https://en.wikipedia.org/wiki/Delhi_High_Court": "Delhi High Court",
    "https://en.wikipedia.org/wiki/Gauhati_High_Court": "Gauhati High Court",
    "https://en.wikipedia.org/wiki/Gujarat_High_Court": "Gujarat High Court",
    "https://en.wikipedia.org/wiki/Himachal_Pradesh_High_Court": "Himachal Pradesh High Court",
    "https://en.wikipedia.org/wiki/High_Court_of_Jammu_and_Kashmir_and_Ladakh": "Jammu and Kashmir & Ladakh High Court",
    "https://en.wikipedia.org/wiki/Jharkhand_High_Court": "Jharkhand High Court",
    "https://en.wikipedia.org/wiki/Karnataka_High_Court": "Karnataka High Court",
    "https://en.wikipedia.org/wiki/Kerala_High_Court": "Kerala High Court",
    "https://en.wikipedia.org/wiki/Madhya_Pradesh_High_Court": "Madhya Pradesh High Court",
    "https://en.wikipedia.org/wiki/Madras_High_Court": "Madras High Court",
    "https://en.wikipedia.org/wiki/Manipur_High_Court": "Manipur High Court",
    "https://en.wikipedia.org/wiki/Meghalaya_High_Court": "Meghalaya High Court",
    "https://en.wikipedia.org/wiki/Orissa_High_Court": "Orissa High Court",
    "https://en.wikipedia.org/wiki/Patna_High_Court": "Patna High Court",
    "https://en.wikipedia.org/wiki/Punjab_and_Haryana_High_Court": "Punjab and Haryana High Court",
    "https://en.wikipedia.org/wiki/Rajasthan_High_Court": "Rajasthan High Court",
    "https://en.wikipedia.org/wiki/Sikkim_High_Court": "Sikkim High Court",
    "https://en.wikipedia.org/wiki/Telangana_High_Court": "Telangana High Court",
    "https://en.wikipedia.org/wiki/Tripura_High_Court": "Tripura High Court",
    "https://en.wikipedia.org/wiki/Uttarakhand_High_Court": "Uttarakhand High Court",
    "https://en.wikipedia.org/wiki/List_of_chief_justices_of_India": "List of Chief Justices of India",
    "https://en.wikipedia.org/wiki/Ministry_of_Law_and_Justice_(India)": "Ministry of Law and Justice (India)",
    "https://en.wikipedia.org/wiki/List_of_current_Indian_governors": "List of Current Indian Governors",
    "https://en.wikipedia.org/wiki/List_of_current_Indian_chief_ministers": "List of Current Indian Chief Ministers"
}

# Initialize an empty list for chunks
Wikipedia_Details= []

for url, name in urls_with_names.items():
    try:
        html_content = fetch_html(url)
        if html_content:  # Proceed only if HTML content is successfully fetched
            print(f"Fetched HTML for URL: {url}")  # Debug print

            extracted_text = extract_main_content(html_content)
            if extracted_text:
                print(f"Extracted content from {url} using extract_main_content.")  # Debug print
            else:
                print(f"Can't scrape content from {url} using the <h> and <p> tags")  # Debug print
                soup = BeautifulSoup(html_content, 'html.parser')
                data = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p'])
                scrap_data = [d.text.strip() for d in data if d.text.strip()]  # Strip and check for empty text
                extracted_text = '\n'.join(scrap_data)

            # Only chunk if there's actual text
            if extracted_text.strip():
                chunked_text = chunk_text(extracted_text, chunk_size=180, overlap=45)
                for i, chunk in enumerate(chunked_text):
                    # Wrap text to ensure it fits within the desired width
                    wrapped_text = textwrap.fill(chunk, width=80)

                    Wikipedia_Details.append(f"{name}:\n{wrapped_text}")
            else:
                print(f"No valid content extracted from {url}.")
        else:
            print(f"Failed to fetch content from {url}")  # Debug print

    except Exception as e:
        print(f"Error fetching or processing {url}: {e}")

    time.sleep(1)




Fetched HTML for URL: https://en.wikipedia.org/wiki/Allahabad_High_Court
Extracted content from https://en.wikipedia.org/wiki/Allahabad_High_Court using extract_main_content.
Fetched HTML for URL: https://en.wikipedia.org/wiki/Andhra_Pradesh_High_Court
Extracted content from https://en.wikipedia.org/wiki/Andhra_Pradesh_High_Court using extract_main_content.
Fetched HTML for URL: https://en.wikipedia.org/wiki/Bombay_High_Court
Extracted content from https://en.wikipedia.org/wiki/Bombay_High_Court using extract_main_content.
Fetched HTML for URL: https://en.wikipedia.org/wiki/Calcutta_High_Court
Extracted content from https://en.wikipedia.org/wiki/Calcutta_High_Court using extract_main_content.
Fetched HTML for URL: https://en.wikipedia.org/wiki/Chhattisgarh_High_Court
Extracted content from https://en.wikipedia.org/wiki/Chhattisgarh_High_Court using extract_main_content.
Fetched HTML for URL: https://en.wikipedia.org/wiki/Delhi_High_Court
Extracted content from https://en.wikipedia.org/

In [ ]:
# Print the chunked texts with meaningful titles
for chunk in Wikipedia_Details:
    print(chunk)
    print("-" * 80)

Allahabad High Court:
Jump to content Contents move to sidebar hide (Top) 1 High court Seats 2 History
3 Principal seat and benches 4 Chief Justice Toggle Chief Justice subsection 4.1
List of chief justices 5 Agra High Court Bench Demand Toggle Agra High Court
Bench Demand subsection 5.1 Highest Number of Pending Cases 5.2 Demands for a
separate state 6 Chief Justice and judges Toggle Chief Justice and judges
subsection 6.1 Reporting and citation 6.2 High Court Service 6.3 Commemorative
postal stamps 7 References Toggle References subsection 7.1 Cited sources 7.2
External links Toggle the table of contents Allahabad High Court 13 languages
अवधी भोजपुरी Deutsch हिन्दी മലയാളം मराठी ଓଡ଼ିଆ ਪੰਜਾਬੀ پنجابی संस्कृतम् Simple
English தமிழ் اردو Edit links Article Talk English Read Edit View history Tools
Tools move to sidebar hide Actions Read Edit View history General What links
here Related changes Upload file Special pages Permanent link Page information
Cite this page Get shortened URL Downl